# US Census Bureau Demographics EDA

This notebook:
1. **Loads** US Census American Community Survey (ACS) 5-year estimates at the **county** level via the Census API.
2. **Explores** demographic variables (population, age, income, race, etc.) by county.
3. Runs **basic exploratory data analysis** (EDA) in Python.

* The Census Bureau provides free API access to ACS and decennial census data. [LINK](https://www.census.gov/data/developers/data-sets.html)

* ACS 5-year estimates provide the most reliable data for smaller geographic areas (counties).

* Target years for this project: 2016–2019 (overlap with pesticide, health, and cropland data).

* Requires: Python with `pandas`, `numpy`, `matplotlib`, `seaborn`, and `requests`.

* **Note:** An API key is optional but recommended for higher rate limits. Get one at [census.gov](https://api.census.gov/data/key_signup.html).

In [ ]:
# Load relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

## 1. Load county-level ACS data via Census API

We use the Census Data API to fetch ACS 5-year estimates. Key variables:
- **B01003_001E**: Total population
- **B01002_001E**: Median age
- **B19013_001E**: Median household income
- **B25077_001E**: Median home value
- **B03002_001E**: Total population (race)
- **B03002_003E**: Not Hispanic/Latino - White alone
- **B03002_004E**: Not Hispanic/Latino - Black alone
- **B03002_005E**: Not Hispanic/Latino - American Indian
- **B03002_006E**: Not Hispanic/Latino - Asian alone
- **B03002_012E**: Hispanic/Latino

In [ ]:
# Census API base URL
CENSUS_BASE = "https://api.census.gov/data"

# ACS 5-year variables for demographics
VARS = [
    "NAME",
    "B01003_001E",   # Total population
    "B01002_001E",   # Median age
    "B19013_001E",   # Median household income
    "B25077_001E",   # Median home value
    "B03002_001E",   # Total population (race)
    "B03002_003E",   # White alone
    "B03002_004E",   # Black alone
    "B03002_005E",   # American Indian alone
    "B03002_006E",   # Asian alone
    "B03002_012E",   # Hispanic/Latino
]

def get_acs_counties(year: int, api_key: str = None) -> pd.DataFrame:
    """Fetch ACS 5-year county-level data for all US counties."""
    url = f"{CENSUS_BASE}/{year}/acs/acs5"
    params = {
        "get": ",".join(VARS),
        "for": "county:*",
        "in": "state:*",
    }
    if api_key:
        params["key"] = api_key
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

# Load 2019 ACS (can change to 2016-2019 for target years)
year = 2019
df = get_acs_counties(year)
df.head(10)

In [ ]:
# Convert numeric columns (Census returns strings)
numeric_cols = [c for c in df.columns if c not in ["NAME", "state", "county"]]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Create 5-digit FIPS for merging with other datasets
df["STATE_FIPS"] = df["state"].astype(str).str.zfill(2)
df["COUNTY_FIPS"] = df["county"].astype(str).str.zfill(3)
df["FIPS"] = df["STATE_FIPS"] + df["COUNTY_FIPS"]

df.head()

In [ ]:
# Inspect structure
print("Columns:", list(df.columns))
print("\nShape:", df.shape)
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing values (Census uses -666666666 etc. for missing/ suppressed)
print("Missing / null values:")
print(df.isnull().sum())
print("\nValues that might indicate suppression (negative or very large):")
for col in numeric_cols:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"  {col}: {neg} negative values")

## 2. Visualizations

Explore distributions of key demographic variables.

In [ ]:
# Population distribution (log scale - many small counties)
pop = df["B01003_001E"].dropna()
pop = pop[pop > 0]  # exclude suppressed
plt.figure(figsize=(10, 4))
plt.hist(np.log10(pop), bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Log10(Population)")
plt.ylabel("Count of Counties")
plt.title("Distribution of County Population (log scale)")
plt.show()

In [ ]:
# Median age distribution
age = df["B01002_001E"].dropna()
age = age[(age > 0) & (age < 120)]
plt.figure(figsize=(8, 4))
plt.hist(age, bins=40, edgecolor="black", alpha=0.7)
plt.xlabel("Median Age)")
plt.ylabel("Count of Counties")
plt.title("Distribution of County Median Age")
plt.show()

In [ ]:
# Median household income distribution
inc = df["B19013_001E"].dropna()
inc = inc[(inc > 0) & (inc < 1e7)]  # exclude suppressed/outliers
plt.figure(figsize=(8, 4))
plt.hist(inc / 1000, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Median Household Income ($1000s)")
plt.ylabel("Count of Counties")
plt.title("Distribution of County Median Household Income")
plt.show()

In [ ]:
# Race/ethnicity breakdown (sample of top 20 pop counties)
top_counties = df.nlargest(20, "B01003_001E")
race_cols = ["B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E"]
race_labels = ["White", "Black", "Asian", "Hispanic"]
total = top_counties["B03002_001E"].replace(0, np.nan)
pct = top_counties[race_cols].div(total, axis=0) * 100
pct.columns = race_labels
pct["NAME"] = top_counties["NAME"].str.split(",").str[0]  # county name only
pct.set_index("NAME").plot(kind="barh", stacked=True, figsize=(10, 8))
plt.title("Race/Ethnicity % - Top 20 Counties by Population")
plt.xlabel("Percentage")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of key numeric variables
key_vars = ["B01003_001E", "B01002_001E", "B19013_001E", "B25077_001E"]
key_labels = ["Population", "Median Age", "Median Income", "Median Home Value"]
corr_df = df[key_vars].replace([-666666666, -222222222], np.nan)
corr_df = corr_df[(corr_df > 0).all(axis=1)]
corr_df.columns = key_labels
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of Key Demographics")
plt.tight_layout()
plt.show()